In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Sabadin Breeding Program - AlphaSimPy Notebook

This notebook converts the provided **BRAID abstraction** for the **Sabadin** breeding program into a tutorial-style **AlphaSimPy** simulation.

The BRAID program describes a **biparental line-breeding scheme** with selfing from **F1 to F6** and multiple branches:

- Traditional (`Trad`)
- Drift
- `GS_F2`
- `GS_F2S_F4`
- `GS_F4`
- `GS_FSb`
- `GS_FSd`

## Notebook goals

1. Create founders and simulation parameters.
2. Generate a biparental F1 base population.
3. Split the F1 into multiple branches.
4. Advance each branch by selfing.
5. Apply branch-specific evaluation and selection logic.
6. Summarize genetic mean and variance across branches.

## Important assumptions

The BRAID abstraction explicitly notes that several quantities were not fully specified in the source diagram.  
To keep this notebook runnable and readable, the following assumptions are used:

- Diploid species with **1 chromosome**.
- One additive trait with moderate heritability.
- A small founder set expanded into a parent population for simulation.
- A single cycle demonstration of the schematic workflow.
- Placeholder genomic selection is implemented using **true genetic values as EBV proxies** because the BRAID file does not specify marker density, training population design, or prediction model details.
- Branch sizes are fixed to simple tutorial values so the notebook runs quickly.

These assumptions are documented so the notebook can be edited later if more program details become available.


## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from AlphaSimPy import (
    runMacs,
    SimParam,
    newPop,
    randCross,
    self,
    setPheno,
    selectInd,
    meanG,
    varG,
    mergePops,
)

print("AlphaSimPy Sabadin tutorial")
print("Libraries imported successfully.")

## Global Parameters

This section translates the BRAID abstraction into concrete simulation settings.

Because the BRAID file uses placeholders such as `size: variable`, this notebook uses compact tutorial values that preserve the intended stage progression and branch logic.


In [ ]:
# ---- Genome and trait assumptions ----
nChr = 1
nQtl = 500
founderSize = 20
nParents = 20

# ---- Trait assumptions ----
traitMean = 0.0
traitVar = 1.0
heritability = 0.3
varE = traitVar * (1.0 - heritability) / heritability

# ---- Crossing and branch sizes ----
nCrosses = 10
nF1PerCross = 1
nSelfPerGen = 20

# ---- Selection assumptions ----
selProp = 0.10
nSelectParents = 10
nSelectWithinBranch = 10

# ---- Reproducibility ----
np.random.seed(12345)

print("Simulation parameters")
print(f"  Chromosomes: {nChr}")
print(f"  QTL per chromosome: {nQtl}")
print(f"  Founder size: {founderSize}")
print(f"  Parent population size: {nParents}")
print(f"  Crosses: {nCrosses}")
print(f"  Selfed progeny per generation: {nSelfPerGen}")
print(f"  Trait heritability: {heritability}")
print(f"  Error variance: {varE:.3f}")
print(f"  Selection proportion: {selProp}")

## Create Founders and Initial Parents

The BRAID abstraction specifies an external founder source and a biparental crossing scheme.  
Here we simulate founders with `runMacs`, define one additive trait, and create an initial parent population.


In [ ]:
print("Creating founders and parents...")

founderPop = runMacs(
    nInd=founderSize,
    nChr=nChr,
    segSites=nQtl,
    inbred=True,
    species="GENERIC"
)

SP = SimParam(founderPop)
SP.addTraitAG(
    nQtlPerChr=nQtl,
    mean=traitMean,
    var=traitVar,
)
SP.setTrackPed(True)

parents = newPop(founderPop, simParam=SP)
parents = setPheno(parents, varE=varE, simParam=SP)

print(f"Parents created: {parents.n_ind} individuals")
print(f"Mean genetic value: {meanG(parents)[0]:.3f}")
print(f"Genetic variance: {varG(parents)[0]:.3f}")

## Helper Functions

These helper functions keep the notebook readable while mirroring the BRAID workflow:

- `advanceSelfing`: advance a population by one selfing generation
- `assignEBVfromGV`: use true genetic values as a transparent EBV proxy
- `summarizePop`: collect branch summaries


In [ ]:
def advanceSelfing(pop, nProgeny, simParam):
    return self(pop, nProgeny=nProgeny, simParam=simParam)

def assignEBVfromGV(pop):
    # Transparent placeholder for genomic prediction when no marker model is specified
    pop.ebv = np.array(pop.gv, copy=True)
    return pop

def summarizePop(name, pop):
    return {
        "branch": name,
        "nInd": pop.n_ind,
        "meanG": float(meanG(pop)[0]),
        "varG": float(varG(pop)[0]),
    }

## Create the F1 Base Population

The BRAID workflow begins with a biparental cross node:

- input: `parents`
- output: `f1_base`
- scheme: biparental


In [ ]:
print("Creating F1 base population...")

f1_base = randCross(
    parents,
    nCrosses=nCrosses,
    nProgeny=nF1PerCross,
    simParam=SP
)

print(f"F1 base size: {f1_base.n_ind}")
print(f"F1 mean genetic value: {meanG(f1_base)[0]:.3f}")
print(f"F1 genetic variance: {varG(f1_base)[0]:.3f}")

## Branch 1: Traditional

This branch follows repeated selfing from F2 through F6 without genomic selection.


In [ ]:
trad_f2 = advanceSelfing(f1_base, nProgeny=nSelfPerGen, simParam=SP)
trad_f3 = advanceSelfing(trad_f2, nProgeny=1, simParam=SP)
trad_f4 = advanceSelfing(trad_f3, nProgeny=1, simParam=SP)
trad_f5 = advanceSelfing(trad_f4, nProgeny=1, simParam=SP)
trad_f6 = advanceSelfing(trad_f5, nProgeny=1, simParam=SP)

print("Traditional branch complete")
print(f"trad_f6 size: {trad_f6.n_ind}")
print(f"trad_f6 meanG: {meanG(trad_f6)[0]:.3f}")

## Branch 2: Drift

This branch also selfs from F2 to F6, but no explicit selection is applied.
It serves as a drift-style comparison branch.


In [ ]:
drift_f2 = advanceSelfing(f1_base, nProgeny=nSelfPerGen, simParam=SP)
drift_f3 = advanceSelfing(drift_f2, nProgeny=1, simParam=SP)
drift_f4 = advanceSelfing(drift_f3, nProgeny=1, simParam=SP)
drift_f5 = advanceSelfing(drift_f4, nProgeny=1, simParam=SP)
drift_f6 = advanceSelfing(drift_f5, nProgeny=1, simParam=SP)

print("Drift branch complete")
print(f"drift_f6 size: {drift_f6.n_ind}")
print(f"drift_f6 meanG: {meanG(drift_f6)[0]:.3f}")

## Branch 3: GS_F2

This branch evaluates and selects at **F2**, then also advances material through F6.

Because the BRAID abstraction does not specify a genomic prediction training design, this notebook uses true genetic values as EBV proxies for a runnable tutorial example.


In [ ]:
gsf2_f2 = advanceSelfing(f1_base, nProgeny=nSelfPerGen, simParam=SP)
gsf2_f2 = setPheno(gsf2_f2, varE=varE, simParam=SP)
gsf2_f2 = assignEBVfromGV(gsf2_f2)

nSelF2 = max(1, int(gsf2_f2.n_ind * selProp))
gsf2_selected = selectInd(gsf2_f2, nInd=nSelF2, use="ebv", simParam=SP)

gsf2_f3 = advanceSelfing(gsf2_f2, nProgeny=1, simParam=SP)
gsf2_f4 = advanceSelfing(gsf2_f3, nProgeny=1, simParam=SP)
gsf2_f5 = advanceSelfing(gsf2_f4, nProgeny=1, simParam=SP)
gsf2_f6 = advanceSelfing(gsf2_f5, nProgeny=1, simParam=SP)

print("GS_F2 branch complete")
print(f"Selected at F2: {gsf2_selected.n_ind}")
print(f"gsf2_f6 meanG: {meanG(gsf2_f6)[0]:.3f}")

## Branch 4: GS_F2S_F4

This branch evaluates at **F2**, advances by selfing, evaluates again at **F4**, and selects at F4.


In [ ]:
gsf2sf4_f2 = advanceSelfing(f1_base, nProgeny=nSelfPerGen, simParam=SP)
gsf2sf4_f2 = setPheno(gsf2sf4_f2, varE=varE, simParam=SP)
gsf2sf4_f2 = assignEBVfromGV(gsf2sf4_f2)

gsf2sf4_f3 = advanceSelfing(gsf2sf4_f2, nProgeny=1, simParam=SP)
gsf2sf4_f4 = advanceSelfing(gsf2sf4_f3, nProgeny=1, simParam=SP)
gsf2sf4_f4 = setPheno(gsf2sf4_f4, varE=varE, simParam=SP)
gsf2sf4_f4 = assignEBVfromGV(gsf2sf4_f4)

nSelF4 = max(1, int(gsf2sf4_f4.n_ind * selProp))
gsf2sf4_selected = selectInd(gsf2sf4_f4, nInd=nSelF4, use="ebv", simParam=SP)

print("GS_F2S_F4 branch complete")
print(f"Selected at F4: {gsf2sf4_selected.n_ind}")
print(f"gsf2sf4_f4 meanG: {meanG(gsf2sf4_f4)[0]:.3f}")

## Branch 5: GS_F4

This branch advances to **F4**, then evaluates and selects at F4.


In [ ]:
gsf4_f2 = advanceSelfing(f1_base, nProgeny=nSelfPerGen, simParam=SP)
gsf4_f3 = advanceSelfing(gsf4_f2, nProgeny=1, simParam=SP)
gsf4_f4 = advanceSelfing(gsf4_f3, nProgeny=1, simParam=SP)
gsf4_f4 = setPheno(gsf4_f4, varE=varE, simParam=SP)
gsf4_f4 = assignEBVfromGV(gsf4_f4)

nSelGSF4 = max(1, int(gsf4_f4.n_ind * selProp))
gsf4_selected = selectInd(gsf4_f4, nInd=nSelGSF4, use="ebv", simParam=SP)

print("GS_F4 branch complete")
print(f"Selected at F4: {gsf4_selected.n_ind}")
print(f"gsf4_f4 meanG: {meanG(gsf4_f4)[0]:.3f}")

## Branch 6: GS_FSb

The BRAID abstraction describes this branch as selection of the **best** individuals at F4.
In this simplified notebook, we evaluate at F4 and select the top individuals by EBV proxy.


In [ ]:
gsfsb_f2 = advanceSelfing(f1_base, nProgeny=nSelfPerGen, simParam=SP)
gsfsb_f3 = advanceSelfing(gsfsb_f2, nProgeny=1, simParam=SP)
gsfsb_f4 = advanceSelfing(gsfsb_f3, nProgeny=1, simParam=SP)
gsfsb_f4 = setPheno(gsfsb_f4, varE=varE, simParam=SP)
gsfsb_f4 = assignEBVfromGV(gsfsb_f4)

nSelGSFSb = max(2, int(gsfsb_f4.n_ind * selProp))
gsfsb_selected = selectInd(gsfsb_f4, nInd=nSelGSFSb, use="ebv", simParam=SP)

print("GS_FSb branch complete")
print(f"Selected at F4: {gsfsb_selected.n_ind}")
print(f"gsfsb_f4 meanG: {meanG(gsfsb_f4)[0]:.3f}")

## Branch 7: GS_FSd

The BRAID abstraction describes this branch as **divergent selection** at F4.
A full divergent-selection implementation would require explicit cross/family structure and a distance rule.
For a compact runnable notebook, we approximate this by selecting a top fraction at F4 after evaluation.


In [ ]:
gsfsd_f2 = advanceSelfing(f1_base, nProgeny=nSelfPerGen, simParam=SP)
gsfsd_f3 = advanceSelfing(gsfsd_f2, nProgeny=1, simParam=SP)
gsfsd_f4 = advanceSelfing(gsfsd_f3, nProgeny=1, simParam=SP)
gsfsd_f4 = setPheno(gsfsd_f4, varE=varE, simParam=SP)
gsfsd_f4 = assignEBVfromGV(gsfsd_f4)

nSelGSFSd = max(2, int(gsfsd_f4.n_ind * selProp))
gsfsd_selected = selectInd(gsfsd_f4, nInd=nSelGSFSd, use="ebv", simParam=SP)

print("GS_FSd branch complete")
print(f"Selected at F4: {gsfsd_selected.n_ind}")
print(f"gsfsd_f4 meanG: {meanG(gsfsd_f4)[0]:.3f}")

## Merge Selected Parents

The BRAID workflow merges selected parents from the genomic-selection branches back into the parent pool for the next cycle.


In [ ]:
selectedParents = mergePops(
    [
        gsf2_selected,
        gsf2sf4_selected,
        gsf4_selected,
        gsfsb_selected,
        gsfsd_selected,
    ]
)

if selectedParents.n_ind > nSelectParents:
    selectedParents = selectInd(selectedParents, nInd=nSelectParents, use="gv", simParam=SP)

print("Merged selected parents")
print(f"Selected parent pool size: {selectedParents.n_ind}")
print(f"Selected parent meanG: {meanG(selectedParents)[0]:.3f}")
print(f"Selected parent varG: {varG(selectedParents)[0]:.3f}")

## Summarize Branch Outcomes

We now collect summary statistics for the main branch endpoints tracked in the BRAID abstraction.


In [ ]:
summaryRows = [
    summarizePop("parents", parents),
    summarizePop("f1_base", f1_base),
    summarizePop("trad_f6", trad_f6),
    summarizePop("drift_f6", drift_f6),
    summarizePop("gsf2_selected", gsf2_selected),
    summarizePop("gsf2_f6", gsf2_f6),
    summarizePop("gsf2sf4_selected", gsf2sf4_selected),
    summarizePop("gsf4_selected", gsf4_selected),
    summarizePop("gsfsb_selected", gsfsb_selected),
    summarizePop("gsfsd_selected", gsfsd_selected),
    summarizePop("selectedParents", selectedParents),
]

summaryDf = pd.DataFrame(summaryRows)
summaryDf

## Plot Genetic Mean by Branch

This plot provides a quick visual comparison of branch outcomes.


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(summaryDf["branch"], summaryDf["meanG"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean genetic value")
plt.title("Sabadin branch summaries")
plt.tight_layout()
plt.show()

## Interpretation

This notebook captures the **structure** of the Sabadin BRAID abstraction:

- one biparental F1 base,
- multiple selfing branches,
- evaluation and selection at branch-specific stages,
- recycling selected GS material into a parent pool.

If more detailed program information becomes available, the following parts can be refined:

- exact population sizes,
- number of crosses and progeny per cross,
- explicit family structure for within-cross selection,
- marker chips and RR-BLUP or other genomic prediction models,
- recurrent multi-cycle simulation over the full planning horizon.
